In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from pathlib import Path

# Reproducibility
torch.set_default_dtype(torch.float32)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Problem parameters
SIGMA = 0.25
RHO = 0.20
LAMBDA = 1.0

# Grid resolution
GRID_SIZE = 100

# Number of optimization steps
STEPS = 1500

# Learning rate
LR = 0.03

# Output directory
OUTPUT_DIR = Path("lamp_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

Using device: cuda


In [ ]:
def create_grid(grid_size=100):
    x = torch.linspace(0, 1, grid_size, device=DEVICE)
    y = torch.linspace(0, 1, grid_size, device=DEVICE)

    xx, yy = torch.meshgrid(x, y, indexing="xy")

    # Shape: (grid_size * grid_size, 2)
    grid = torch.stack([xx.flatten(), yy.flatten()], dim=1)

    return grid, xx, yy


grid, grid_x, grid_y = create_grid(GRID_SIZE)

print("Grid shape:", grid.shape)

Grid shape: torch.Size([10000, 2])


In [ ]:
def brightness_map(lamps, grid, sigma=SIGMA):
    """
    Compute total brightness at every point in the room.

    lamps: (N, 2)
    grid:  (M, 2)

    returns:
        brightness: (M,)
    """

    # grid[:, None, :] -> (M, 1, 2)
    # lamps[None, :, :] -> (1, N, 2)
    differences = grid[:, None, :] - lamps[None, :, :]

    # Squared Euclidean distance
    squared_distances = torch.sum(differences ** 2, dim=2)

    # Gaussian brightness
    individual_brightness = torch.exp(
        -squared_distances / (2 * sigma ** 2)
    )

    # Add brightness from all lamps
    total_brightness = individual_brightness.sum(dim=1)

    return total_brightness

In [ ]:
def repulsion(lamps, rho=RHO):
    """
    Gaussian repulsion between every pair of lamps.
    """

    n = lamps.shape[0]

    # Pairwise differences
    differences = lamps[:, None, :] - lamps[None, :, :]

    # Pairwise squared distances
    squared_distances = torch.sum(differences ** 2, dim=2)

    # Gaussian repulsion
    values = torch.exp(
        -squared_distances / (2 * rho ** 2)
    )

    # Only take i < j
    mask = torch.triu(
        torch.ones(n, n, device=lamps.device, dtype=torch.bool),
        diagonal=1
    )

    return values[mask].sum()

In [ ]:
def objective(lamps, grid, sigma=SIGMA, rho=RHO,lam=LAMBDA):
    """
    Full objective:

        average brightness - lambda * repulsion
    """

    brightness = brightness_map(
        lamps,
        grid,
        sigma=sigma
    )

    average_brightness = brightness.mean()

    repulsion_value = repulsion(
        lamps,
        rho=rho
    )

    J = average_brightness - lam * repulsion_value

    return J, average_brightness, repulsion_value

In [ ]:
def initialize_lamps(num_lamps=3, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)

    lamps = torch.rand(
        num_lamps,
        2,
        device=DEVICE
    )

    lamps.requires_grad_(True)

    return lamps

In [ ]:
def optimize_lamps(num_lamps=3,seed=0,steps=1500,lr=0.03,sigma=SIGMA,rho=RHO,lam=LAMBDA,save_every=10
):
    """
    Optimize lamp positions using Adam.
    """

    lamps = initialize_lamps(
        num_lamps=num_lamps,
        seed=seed
    )

    # Adam optimizer
    optimizer = torch.optim.Adam(
        [lamps],
        lr=lr
    )

    # Store optimization history for GIFs
    history = []

    objective_history = []
    brightness_history = []
    repulsion_history = []

    for step in range(steps):

        optimizer.zero_grad()

        J, avg_brightness, repulsion_value = objective(
            lamps,
            grid,
            sigma=sigma,
            rho=rho,
            lam=lam
        )

        # Optimizers minimize, so minimize -J
        loss = -J

        loss.backward()
        optimizer.step()

        # Keep lamps inside the unit square
        with torch.no_grad():
            lamps.clamp_(0.0, 1.0)

        # Save trajectory
        if step % save_every == 0 or step == steps - 1:
            history.append(
                lamps.detach().cpu().numpy().copy()
            )

        objective_history.append(J.item())
        brightness_history.append(avg_brightness.item())
        repulsion_history.append(repulsion_value.item())

    return {
        "lamps": lamps.detach().cpu().numpy(),
        "history": history,
        "objective": objective_history,
        "brightness": brightness_history,
        "repulsion": repulsion_history,
    }

In [ ]:
def get_brightness_map(lamps):
    """
    Compute brightness and reshape it into a 2D room map.
    """

    with torch.no_grad():
        brightness = brightness(
            lamps,
            grid,
        )

    brightness_map = brightness_map.reshape(
        GRID_SIZE,
        GRID_SIZE
    ).cpu().numpy()

    return brightness_map

In [21]:
def plot_brightness_map(
    lamps,
    title,
    filename=None,
    show_lamps=True
):
    """
    Plot the brightness of the room.
    """

    brightness_map = get_brightness_map(lamps)

    plt.figure(figsize=(6, 6))

    plt.imshow(
        brightness_map.T,
        origin="lower",
        extent=[0, 1, 0, 1],
        aspect="equal"
    )

    if show_lamps:
        lamp_positions = lamps.detach().cpu().numpy()

        plt.scatter(
            lamp_positions[:, 0],
            lamp_positions[:, 1],
            s=100,
            c="red",
            marker="*",
            edgecolors="black",
            linewidths=0.8,
            label="Lamps"
        )

        plt.legend()

    plt.colorbar(
        label="Brightness"
    )

    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(title)
    plt.tight_layout()

    if filename is not None:
        plt.savefig(
            filename,
            dpi=150,
            bbox_inches="tight"
        )

    plt.show()

In [22]:
def plot_before_after(
    initial_lamps,
    final_lamps,
    filename=None
):
    """
    Plot brightness maps before and after optimization.
    """

    initial_map = get_brightness_map(
        initial_lamps
    )

    final_map = get_brightness_map(
        final_lamps
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 5)
    )

    # -------------------------
    # Before
    # -------------------------
    im1 = axes[0].imshow(
        initial_map.T,
        origin="lower",
        extent=[0, 1, 0, 1],
        aspect="equal"
    )

    initial_np = initial_lamps.cpu().numpy()

    axes[0].scatter(
        initial_np[:, 0],
        initial_np[:, 1],
        s=100,
        c="red",
        marker="*",
        edgecolors="black"
    )

    axes[0].set_title("Before Optimization")
    axes[0].set_xlabel("x")
    axes[0].set_ylabel("y")

    fig.colorbar(
        im1,
        ax=axes[0],
        fraction=0.046
    )

    # -------------------------
    # After
    # -------------------------
    im2 = axes[1].imshow(
        final_map.T,
        origin="lower",
        extent=[0, 1, 0, 1],
        aspect="equal"
    )

    final_np = final_lamps.cpu().numpy()

    axes[1].scatter(
        final_np[:, 0],
        final_np[:, 1],
        s=100,
        c="red",
        marker="*",
        edgecolors="black"
    )

    axes[1].set_title("After Optimization")
    axes[1].set_xlabel("x")
    axes[1].set_ylabel("y")

    fig.colorbar(
        im2,
        ax=axes[1],
        fraction=0.046
    )

    plt.tight_layout()

    if filename is not None:
        plt.savefig(
            filename,
            dpi=150,
            bbox_inches="tight"
        )

    plt.show()


In [23]:
def create_optimization_gif(
    history,
    filename,
    fps=20
):
    """
    Create a GIF showing lamps moving during optimization.
    """

    fig, ax = plt.subplots(
        figsize=(6, 6)
    )

    # Initial frame
    first_lamps = history[0]["lamps"]

    brightness = get_brightness_map(
        torch.tensor(
            first_lamps,
            dtype=torch.float32,
            device=DEVICE
        )
    )

    image = ax.imshow(
        brightness.T,
        origin="lower",
        extent=[0, 1, 0, 1],
        aspect="equal",
        vmin=0,
        vmax=3
    )

    scatter = ax.scatter(
        first_lamps[:, 0],
        first_lamps[:, 1],
        s=100,
        c="red",
        marker="*",
        edgecolors="black"
    )

    title = ax.set_title(
        "Lamp Optimization"
    )

    ax.set_xlabel("x")
    ax.set_ylabel("y")

    fig.colorbar(
        image,
        ax=ax,
        label="Brightness"
    )

    # --------------------------------------------------------
    # Update function
    # --------------------------------------------------------

    def update(frame):

        lamps_np = history[frame]["lamps"]

        lamps_tensor = torch.tensor(
            lamps_np,
            dtype=torch.float32,
            device=DEVICE
        )

        brightness = get_brightness_map(
            lamps_tensor
        )

        image.set_data(
            brightness.T
        )

        scatter.set_offsets(
            lamps_np
        )

        title.set_text(
            f"Step {frame + 1}/{len(history)} | "
            f"J = {history[frame]['objective']:.4f}"
        )

        return image, scatter, title

    # --------------------------------------------------------
    # Animate
    # --------------------------------------------------------

    animation = FuncAnimation(
        fig,
        update,
        frames=len(history),
        interval=50,
        blit=False
    )

    animation.save(
        filename,
        writer=PillowWriter(fps=fps)
    )

    plt.close(fig)

    print(f"Saved GIF: {filename}")

In [24]:
NUM_LAMPS = 3
LEARNING_RATE = 0.03

print("=" * 60)
print("MAIN EXPERIMENT: 3 LAMPS")
print("=" * 60)

# Use one random seed for the main run.
MAIN_SEED = 0

# Initial positions
initial_lamps = initialize_lamps(
    NUM_LAMPS,
    seed=MAIN_SEED
).detach()

print("\nInitial lamp positions:")
print(initial_lamps.cpu().numpy())

# Optimize
final_lamps, history = optimize_lamps(
    num_lamps=NUM_LAMPS,
    seed=MAIN_SEED,
    steps=STEPS,
    learning_rate=LEARNING_RATE
)

print("\nFinal lamp positions:")
print(final_lamps.cpu().numpy())

# Final objective
final_objective, final_brightness, final_repulsion = (
    objective(
        final_lamps,
        grid
    )
)

print("\nFinal results:")
print(f"Average brightness : {final_brightness.item():.6f}")
print(f"Repulsion          : {final_repulsion.item():.6f}")
print(f"Objective J        : {final_objective.item():.6f}")

MAIN EXPERIMENT: 3 LAMPS

Initial lamp positions:
[[0.39904648 0.51667917]
 [0.02493039 0.9400794 ]
 [0.9458541  0.79673123]]


TypeError: optimize_lamps() got an unexpected keyword argument 'learning_rate'